# Módulo 2 — Modelação e avaliação de desempenho

Este notebook corresponde ao Módulo 2 do projeto. O objetivo é treinar e avaliar diferentes modelos de classificação para prever a readmissão hospitalar em menos de 30 dias.

Serão treinados dois modelos interpretáveis — Regressão Logística e Árvore de Decisão — e um modelo menos interpretável — Random Forest.

Os modelos serão comparados com métricas adequadas para classificação, incluindo accuracy, precision, recall, F1-score e AUC. Será dada atenção especial ao desempenho na classe positiva, correspondente aos pacientes readmitidos em menos de 30 dias, uma vez que esta classe é minoritária.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

In [4]:
DATA_PATH = Path("../outputs/dados/diabetic_data_clean.csv")

df_model = pd.read_csv(DATA_PATH)

df_model.shape

(101766, 48)

## 2.2 Separação das variáveis explicativas e da variável-alvo

Nesta etapa são separadas as variáveis explicativas da variável-alvo. A variável a prever é `readmitted_30`, que indica se o paciente foi readmitido em menos de 30 dias.

São excluídos os identificadores `encounter_id` e `patient_nbr`, por não representarem características generalizáveis. Também são removidas `readmitted` e `readmitted_30` do conjunto de variáveis explicativas, para evitar fuga de informação.

In [7]:
# Algumas variáveis são códigos administrativos, por isso serão tratadas como categóricas
coded_categorical_cols = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]

for col in coded_categorical_cols:
    if col in df_model.columns:
        df_model[col] = df_model[col].astype("object")

# Definir variável-alvo
target = "readmitted_30"

# Colunas a excluir das variáveis explicativas
cols_to_exclude = [
    "encounter_id",
    "patient_nbr",
    "readmitted",
    "readmitted_30"
]

X = df_model.drop(columns=cols_to_exclude)
y = df_model[target]

print("Dimensão de X:", X.shape)
print("Dimensão de y:", y.shape)

Dimensão de X: (101766, 44)
Dimensão de y: (101766,)


In [8]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Número de variáveis numéricas:", len(numeric_features))
print("Número de variáveis categóricas:", len(categorical_features))

print("\nVariáveis numéricas:")
print(numeric_features)

print("\nVariáveis categóricas:")
print(categorical_features)

Número de variáveis numéricas: 8
Número de variáveis categóricas: 36

Variáveis numéricas:
['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Variáveis categóricas:
['race', 'gender', 'age', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']


/var/folders/q8/07dvr7w958lgfy1598ltqybm0000gn/T/ipykernel_17465/2190272550.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(include=["object"]).columns.tolist()


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

print("\nDistribuição da variável-alvo no treino:")
display(y_train.value_counts(normalize=True) * 100)

print("\nDistribuição da variável-alvo no teste:")
display(y_test.value_counts(normalize=True) * 100)

X_train: (81412, 44)
X_test: (20354, 44)
y_train: (81412,)
y_test: (20354,)

Distribuição da variável-alvo no treino:


readmitted_30
0    88.839483
1    11.160517
Name: proportion, dtype: float64


Distribuição da variável-alvo no teste:


readmitted_30
0    88.842488
1    11.157512
Name: proportion, dtype: float64

## 2.5 Pipeline de pré-processamento

Nesta etapa é criado um pipeline de pré-processamento para preparar os dados antes do treino dos modelos.

As variáveis numéricas são normalizadas com `StandardScaler`, enquanto as variáveis categóricas são transformadas com `OneHotEncoder`. Esta abordagem permite usar no mesmo modelo variáveis numéricas e categóricas, evitando também fuga de informação, uma vez que o pré-processamento será ajustado apenas ao conjunto de treino.

In [10]:
# Transformador para variáveis numéricas
numeric_transformer = StandardScaler()

# Transformador para variáveis categóricas
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

# Pipeline de pré-processamento
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [11]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("X_train original:", X_train.shape)
print("X_train preparado:", X_train_prepared.shape)

print("X_test original:", X_test.shape)
print("X_test preparado:", X_test_prepared.shape)

X_train original: (81412, 44)
X_train preparado: (81412, 2328)
X_test original: (20354, 44)
X_test preparado: (20354, 2328)


## 2.6 Modelo interpretável 1 — Regressão Logística

A Regressão Logística foi utilizada como primeiro modelo interpretável. Este modelo é adequado como referência inicial porque permite estimar a relação entre as variáveis explicativas e a probabilidade de readmissão em menos de 30 dias.

Como a classe positiva é minoritária, foi utilizado o parâmetro `class_weight="balanced"`, para compensar parcialmente o desequilíbrio entre classes.

In [12]:
log_reg_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

log_reg_model.fit(X_train, y_train)

y_pred_logreg = log_reg_model.predict(X_test)
y_proba_logreg = log_reg_model.predict_proba(X_test)[:, 1]

print("Matriz de confusão:")
print(confusion_matrix(y_test, y_pred_logreg))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred_logreg))

print("\nAUC:")
print(roc_auc_score(y_test, y_proba_logreg))

Matriz de confusão:
[[12146  5937]
 [  969  1302]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.93      0.67      0.78     18083
           1       0.18      0.57      0.27      2271

    accuracy                           0.66     20354
   macro avg       0.55      0.62      0.53     20354
weighted avg       0.84      0.66      0.72     20354


AUC:
0.6699424029220123


Leitura crítica

O modelo identifica 57% dos pacientes readmitidos em menos de 30 dias. Isto é positivo para um primeiro modelo, porque a classe positiva é minoritária.

Mas a precisão é baixa: só 18% dos casos classificados como risco de readmissão precoce eram de facto positivos. Ou seja, há muitos falsos positivos.

## 2.7 Modelo interpretável 2 — Árvore de Decisão

A Árvore de Decisão foi utilizada como segundo modelo interpretável. Este tipo de modelo permite representar o processo de decisão através de regras hierárquicas, sendo mais fácil de explicar do que modelos de caixa negra.

Tal como na Regressão Logística, foi usado `class_weight="balanced"` para considerar o desequilíbrio entre a classe negativa e a classe positiva.

In [13]:
tree_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=50,
        class_weight="balanced",
        random_state=42
    ))
])

tree_model.fit(X_train, y_train)

y_pred_tree = tree_model.predict(X_test)
y_proba_tree = tree_model.predict_proba(X_test)[:, 1]

print("Matriz de confusão:")
print(confusion_matrix(y_test, y_pred_tree))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred_tree))

print("\nAUC:")
print(roc_auc_score(y_test, y_proba_tree))

Matriz de confusão:
[[12501  5582]
 [ 1046  1225]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.92      0.69      0.79     18083
           1       0.18      0.54      0.27      2271

    accuracy                           0.67     20354
   macro avg       0.55      0.62      0.53     20354
weighted avg       0.84      0.67      0.73     20354


AUC:
0.6611016431327602


A Árvore de Decisão teve uma accuracy ligeiramente superior, mas teve menor recall na classe positiva. Como o nosso problema é identificar readmissões em menos de 30 dias, o recall da classe 1 é muito importante. Por isso, nesta fase, a Regressão Logística continua ligeiramente mais interessante do ponto de vista preditivo, embora a Árvore de Decisão seja visualmente mais explicável.

## 2.8 Modelo menos interpretável — Random Forest

O Random Forest foi utilizado como modelo menos interpretável. Este algoritmo combina várias árvores de decisão, podendo captar relações mais complexas entre variáveis.

Embora seja menos transparente do que uma árvore individual ou uma regressão logística, pode apresentar melhor desempenho preditivo em problemas com muitas variáveis e relações não lineares.

Foi usado `class_weight="balanced"` para considerar o desequilíbrio entre a classe negativa e a classe positiva.

In [14]:
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print("Matriz de confusão:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred_rf))

print("\nAUC:")
print(roc_auc_score(y_test, y_proba_rf))

Matriz de confusão:
[[11423  6660]
 [  854  1417]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.93      0.63      0.75     18083
           1       0.18      0.62      0.27      2271

    accuracy                           0.63     20354
   macro avg       0.55      0.63      0.51     20354
weighted avg       0.85      0.63      0.70     20354


AUC:
0.6745345895496846


## 2.9 Comparação dos modelos

Nesta etapa são comparados os três modelos treinados: Regressão Logística, Árvore de Decisão e Random Forest.

A comparação é feita com base em accuracy, precision, recall, F1-score e AUC. Dado que a classe positiva é minoritária, é dada especial atenção ao recall e ao F1-score da classe 1.

In [15]:
model_results = pd.DataFrame({
    "modelo": [
        "Regressão Logística",
        "Árvore de Decisão",
        "Random Forest"
    ],
    "accuracy": [
        accuracy_score(y_test, y_pred_logreg),
        accuracy_score(y_test, y_pred_tree),
        accuracy_score(y_test, y_pred_rf)
    ],
    "precision_classe_1": [
        precision_score(y_test, y_pred_logreg),
        precision_score(y_test, y_pred_tree),
        precision_score(y_test, y_pred_rf)
    ],
    "recall_classe_1": [
        recall_score(y_test, y_pred_logreg),
        recall_score(y_test, y_pred_tree),
        recall_score(y_test, y_pred_rf)
    ],
    "f1_classe_1": [
        f1_score(y_test, y_pred_logreg),
        f1_score(y_test, y_pred_tree),
        f1_score(y_test, y_pred_rf)
    ],
    "auc": [
        roc_auc_score(y_test, y_proba_logreg),
        roc_auc_score(y_test, y_proba_tree),
        roc_auc_score(y_test, y_proba_rf)
    ]
})

model_results

,modelo,accuracy,precision_classe_1,recall_classe_1,f1_classe_1,auc
0,Regressão Logística,0.660706,0.179859,0.573316,0.273817,0.669942
1,Árvore de Decisão,0.674364,0.179962,0.539410,0.269883,0.661102
2,Random Forest,0.630834,0.175436,0.623954,0.273869,0.674535


In [16]:
model_results.to_csv("../outputs/tabelas/comparacao_modelos_modulo2.csv", index=False)

## 2.10 Afinação de hiperparâmetros do modelo selecionado

Com base na comparação inicial dos modelos, foi selecionado o Random Forest para afinação de hiperparâmetros. Esta escolha deve-se ao facto de este modelo ter apresentado o melhor recall para a classe positiva e a melhor AUC entre os modelos testados.

Foi utilizado Random Search para procurar uma combinação de hiperparâmetros mais adequada, reduzindo o custo computacional face a uma pesquisa exaustiva com Grid Search.

In [17]:
from sklearn.model_selection import RandomizedSearchCV

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

param_distributions = {
    "classifier__n_estimators": [100, 150, 200],
    "classifier__max_depth": [8, 10, 12, 15],
    "classifier__min_samples_leaf": [10, 20, 30, 50],
    "classifier__min_samples_split": [2, 5, 10]
}

random_search_rf = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_distributions,
    n_iter=10,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=-1,
    verbose=1
)

random_search_rf.fit(X_train, y_train)

print("Melhores parâmetros:")
print(random_search_rf.best_params_)

print("\nMelhor score médio em validação cruzada:")
print(random_search_rf.best_score_)

Fitting 3 folds for each of 10 candidates, totalling 30 fits


/Users/franciscovaz/Desktop/UC8_exp_reg_diabetes/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/franciscovaz/Desktop/UC8_exp_reg_diabetes/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/franciscovaz/Desktop/UC8_exp_reg_diabetes/.venv/lib/python3.14/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib wor

Melhores parâmetros:
{'classifier__n_estimators': 150, 'classifier__min_samples_split': 10, 'classifier__min_samples_leaf': 30, 'classifier__max_depth': 12}

Melhor score médio em validação cruzada:
0.2645875389247524


In [18]:
best_rf_model = random_search_rf.best_estimator_

y_pred_best_rf = best_rf_model.predict(X_test)
y_proba_best_rf = best_rf_model.predict_proba(X_test)[:, 1]

print("Matriz de confusão:")
print(confusion_matrix(y_test, y_pred_best_rf))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred_best_rf))

print("\nAUC:")
print(roc_auc_score(y_test, y_proba_best_rf))

Matriz de confusão:
[[11238  6845]
 [  822  1449]]

Relatório de classificação:
              precision    recall  f1-score   support

           0       0.93      0.62      0.75     18083
           1       0.17      0.64      0.27      2271

    accuracy                           0.62     20354
   macro avg       0.55      0.63      0.51     20354
weighted avg       0.85      0.62      0.69     20354


AUC:
0.676122063795416


In [19]:
final_model_results = pd.DataFrame({
    "modelo": [
        "Regressão Logística",
        "Árvore de Decisão",
        "Random Forest",
        "Random Forest afinado"
    ],
    "accuracy": [
        accuracy_score(y_test, y_pred_logreg),
        accuracy_score(y_test, y_pred_tree),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_best_rf)
    ],
    "precision_classe_1": [
        precision_score(y_test, y_pred_logreg),
        precision_score(y_test, y_pred_tree),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_best_rf)
    ],
    "recall_classe_1": [
        recall_score(y_test, y_pred_logreg),
        recall_score(y_test, y_pred_tree),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_best_rf)
    ],
    "f1_classe_1": [
        f1_score(y_test, y_pred_logreg),
        f1_score(y_test, y_pred_tree),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_best_rf)
    ],
    "auc": [
        roc_auc_score(y_test, y_proba_logreg),
        roc_auc_score(y_test, y_proba_tree),
        roc_auc_score(y_test, y_proba_rf),
        roc_auc_score(y_test, y_proba_best_rf)
    ]
})

final_model_results

,modelo,accuracy,precision_classe_1,recall_classe_1,f1_classe_1,auc
0,Regressão Logística,0.660706,0.179859,0.573316,0.273817,0.669942
1,Árvore de Decisão,0.674364,0.179962,0.539410,0.269883,0.661102
2,Random Forest,0.630834,0.175436,0.623954,0.273869,0.674535
3,Random Forest afinado,0.623317,0.174705,0.638045,0.274302,0.676122


In [20]:
final_model_results.to_csv("../outputs/tabelas/comparacao_modelos_modulo2_final.csv", index=False)